# OneTake Compact Seq2Seq Notice Parser (`Flan-T5-Small` Pruned + 4-Bit QAT → Core ML)

Run all cells on a **Colab Pro A100 / L4 GPU** runtime. Total runtime: ~20 minutes.
At the end, download `OneTakeParserCoreML_Bundle.zip` and extract into `OneTake/MLModels/`.


In [ ]:
!nvidia-smi
!pip install -q "transformers>=4.44.0" sentencepiece accelerate "coremltools>=8.0" "scikit-learn<1.6"


In [ ]:
import os, sys
os.makedirs('/content/onetake_ml', exist_ok=True)
sys.path.insert(0, '/content/onetake_ml')

with open('/content/onetake_ml/dataset_generator.py', 'w', encoding='utf-8') as f:
    f.write('"""Synthetic & adversarial casting notice dataset generator with strict Swift-parity grounding."""\n\nfrom __future__ import annotations\n\nimport random\nimport re\nimport unicodedata\nfrom typing import Optional\n\nPROMPT_PREFIX = "parse audition notice:\\n"\n_STRIP_CHARS = " \\t\\"\'.,;:-"\n\n\ndef _fold_diacritics(text: str) -> str:\n    nfkd = unicodedata.normalize("NFKD", text)\n    return "".join(ch for ch in nfkd if not unicodedata.combining(ch)).lower()\n\n\ndef is_grounded_notice_value(value: Optional[str], source: str) -> bool:\n    """Matches `OnDeviceIntelligence.isGroundedNoticeValue(_:in:)` word-for-word."""\n    if not value:\n        return False\n    candidate = value.strip(_STRIP_CHARS)\n    if len(candidate) < 2 or candidate.lower() == "none":\n        return False\n    folded_candidate = _fold_diacritics(candidate)\n    folded_source = _fold_diacritics(source)\n    if folded_candidate in folded_source:\n        return True\n    words = [w for w in re.split(r"[^a-z0-9]+", candidate.lower()) if w]\n    if not words:\n        return False\n    normalized_source = source.lower()\n    matches = sum(1 for w in words if w in normalized_source)\n    return (matches / len(words)) >= 0.75\n\n\ndef format_tagged_target(\n    project: Optional[str],\n    role: Optional[str],\n    desc: Optional[str],\n    deadline: Optional[str],\n    email: Optional[str],\n) -> str:\n    """Formats the 5 target fields into strict single-line tagged format."""\n    def clean(v: Optional[str]) -> str:\n        if not v:\n            return "NONE"\n        s = " ".join(v.strip().split())\n        s = (\n            s.replace("\\u2013", "-")\n            .replace("\\u2014", "-")\n            .replace("\\u2212", "-")\n            .replace("\\u2018", "\'")\n            .replace("\\u2019", "\'")\n            .replace("\\u201c", \'"\')\n            .replace("\\u201d", \'"\')\n            .replace("\\u2026", "...")\n            .replace("\\u20b9", "Rs.")\n        )\n        return s if s else "NONE"\n\n    return "\\n".join(\n        [\n            f"PROJECT: {clean(project)}",\n            f"ROLE: {clean(role)}",\n            f"DESC: {clean(desc)}",\n            f"DEADLINE: {clean(deadline)}",\n            f"EMAIL: {clean(email)}",\n        ]\n    )\n\n\n_PROJECTS = [\n    "MCCC - KOSHISHE",\n    "PHOTON",\n    "The Quiet Harbor",\n    "Neon Horizon",\n    "Midnight Express",\n    "Monsoon Shadows",\n    "Kala Pani",\n    "Glass Harbor",\n    "Echoes of Dawn",\n    "Saffron & Rain",\n    "Untold Verses",\n    "Red Signal",\n    "Dilli 1998",\n    "Paper Boats",\n    "The Last Audition",\n    "Bombay Noir",\n    "Silver Lining",\n    "Naya Savera",\n    "Starlight Diner",\n    "Vanguard 7",\n]\n\n_STUDIOS = [\n    "Yash Raj Films",\n    "Casting Bay - Casting Studio",\n    "Mukesh Chhabra Casting Co.",\n    "Dharma Talent",\n    "Excel Entertainment",\n    "A24 Indie Unit",\n    "Pinewood Casting",\n    "Viacom18 Studio",\n    "Balaji Telefilms",\n    "Red Chillies Casting",\n]\n\n_ROLES = [\n    "Male Lead",\n    "Female Lead",\n    "Female Talent",\n    "Male Talent",\n    "DETECTIVE COLE",\n    "MAYA",\n    "AARAV",\n    "Supporting Lead",\n    "Character Actor",\n    "Young Father",\n    "College Best Friend",\n    "Senior Inspector",\n    "Rohan (Second Lead)",\n    "Grandmother",\n    "Lead Vocalist",\n]\n\n_DESCS = [\n    "Women aged 20–25 years with strong screen presence and natural expressions",\n    "Men aged 22–25 years, fluent in Hindi & English, freshers welcome",\n    "25–32 yrs, sharp comic timing, athletic build, Mumbai based preferred",\n    "Age 18–24, South Asian look, comfortable with emotional monologue & Hindi dialogue",\n    "30–40 years, theater background preferred, fluent English & conversational Marathi",\n    "20–28 yrs old, natural improviser, guitar skills a plus",\n    "Age 45–55, weathered authoritative voice, shoot in Delhi & Mumbai",\n    "Fresh female faces 19–24 years for upcoming Hindi theatrical feature film",\n]\n\n_DEADLINES = [\n    "15 December, 2026",\n    "30 November 2026",\n    "December 15, 2026 at 5:00 PM",\n    "30/11/2026 17:00",\n    "Nov 30, 2026 5:00 PM",\n    "2026-12-15",\n    "18 Jan 2027",\n    "Monday, 24th Nov 6 PM",\n    "05/02/2027",\n    "12 March 2027 EOD",\n]\n\n_SHOOT_DATES = [\n    "10 January 2027",\n    "February 2027 (25 days schedule)",\n    "02/03/2027 – 18/03/2027",\n    "mid-April 2027",\n]\n\n_SUBMISSION_EMAILS = [\n    "abhinav@mcc.in",\n    "casting@schoolofdrama.com",\n    "profilesforlead@gmail.com",\n    "casting@neonfilm.com",\n    "auditions@midnight.org",\n    "selftape@monsoonfilms.in",\n    "talent.mumbai@castingbay.co.in",\n    "submit.koshishe@gmail.com",\n    "submissions@harborpictures.com",\n]\n\n_DECOY_EMAILS = [\n    "agent@talentagency.com",\n    "info@studiooffice.in",\n    "press@productionhouse.com",\n    "queries@mumbaifilms.org",\n    "desk@management.co.uk",\n]\n\n\ndef _pick(rng: random.Random, items: list[str]) -> str:\n    return items[rng.randrange(len(items))]\n\n\n_TITLE_ADJ = ["Broken", "Velvet", "Silent", "Crimson", "Golden", "Last", "Parallel", "Copper", "Distant", "Wandering", "Urban", "Hidden", "Wild", "Electric"]\n_TITLE_NOUN = ["Monsoon", "Platform", "Echo", "Bridge", "Compass", "Cinema", "Notebook", "Harbor", "Station", "Courtyard", "Skyline", "Letter", "Lantern", "Waves"]\n_ROLE_ADJ = ["Young", "Senior", "Supporting", "Lead", "Primary", "Rookie", "Charming", "Ruthless", "Soft-spoken"]\n_ROLE_NOUN = ["Detective", "Journalist", "Doctor", "Architect", "Musician", "Protagonist", "Antagonist", "Lawyer", "Professor", "Dancer"]\n_CITIES = ["Mumbai", "Delhi", "Bengaluru", "Kolkata", "Pune", "Jaipur", "Hyderabad", "London"]\n_MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]\n_DOMAINS = ["castingunit.in", "filmcraft.co.in", "studionext.com", "indiefilms.org", "talentdesk.in", "selftapes.com"]\n\n\ndef _gen_project(rng: random.Random) -> str:\n    if rng.random() < 0.45:\n        return _pick(rng, _PROJECTS)\n    mode = rng.randrange(3)\n    if mode == 0:\n        return f"{_pick(rng, _TITLE_ADJ)} {_pick(rng, _TITLE_NOUN)}"\n    if mode == 1:\n        tag = "".join(rng.choice("ABCDEFGHIJKLMNOPQRSTUVWXYZ") for _ in range(rng.randint(3, 5)))\n        return f"{tag} - {_pick(rng, _TITLE_NOUN).upper()}"\n    return f"The {_pick(rng, _TITLE_ADJ)} {_pick(rng, _TITLE_NOUN)} S{rng.randint(1, 3)}"\n\n\ndef _gen_role(rng: random.Random) -> str:\n    if rng.random() < 0.45:\n        return _pick(rng, _ROLES)\n    if rng.random() < 0.5:\n        return f"{_pick(rng, _ROLE_ADJ)} {_pick(rng, _ROLE_NOUN)}"\n    first = rng.choice(["Aryan", "Meera", "Kabir", "Naina", "Vikram", "Tara", "Dev", "Zoya", "Rhea", "Samar"])\n    return f"{first} ({rng.choice([\'Lead\', \'Parallel Lead\', \'Supporting\'])})"\n\n\ndef _gen_desc(rng: random.Random) -> str:\n    if rng.random() < 0.45:\n        return _pick(rng, _DESCS)\n    low = rng.randint(18, 45)\n    high = low + rng.randint(4, 10)\n    dash = rng.choice(["-", "\\u2013"])\n    gender = rng.choice(["Male", "Female", "Any gender", "Actors"])\n    trait = rng.choice([\n        "strong screen presence",\n        "fluent Hindi & English",\n        "theater background preferred",\n        "natural expressions",\n        "comic timing & improvisation",\n    ])\n    return f"{gender}, age {low}{dash}{high} yrs, {trait}, shoot in {_pick(rng, _CITIES)}"\n\n\ndef _gen_deadline(rng: random.Random) -> str:\n    if rng.random() < 0.45:\n        return _pick(rng, _DEADLINES)\n    day = rng.randint(1, 28)\n    mon = _pick(rng, _MONTHS)\n    yr = rng.choice([2026, 2027])\n    hr = rng.choice(["5 PM", "6:00 PM", "11:59 PM", "EOD"])\n    return rng.choice([\n        f"{day} {mon}, {yr}",\n        f"{mon[:3]} {day}, {yr} {hr}",\n        f"{day:02d}/{rng.randint(1, 12):02d}/{yr}",\n        f"{yr}-{rng.randint(1, 12):02d}-{day:02d}",\n    ])\n\n\ndef _gen_email(rng: random.Random) -> str:\n    if rng.random() < 0.45:\n        return _pick(rng, _SUBMISSION_EMAILS)\n    prefix = rng.choice(["cast", "audition", "submissions", "talent", "projects", "team"])\n    mid = rng.choice(["", ".mumbai", ".casting", "2026"])\n    return f"{prefix}{mid}@{_pick(rng, _DOMAINS)}"\n\n\ndef _ocr_jitter(rng: random.Random, text: str) -> str:\n    lines = text.splitlines()\n    out = []\n    for line in lines:\n        if rng.random() < 0.15:\n            line = line.replace(":", " : ").replace("  ", " ")\n        if rng.random() < 0.10:\n            line = f"• {line}"\n        out.append(line)\n    joined = "\\n".join(out)\n    if rng.random() < 0.12:\n        joined = joined.replace("Role:", "ROLE -").replace("Deadline:", "Last Date:")\n    return joined\n\n\ndef _build_sample(rng: random.Random, category: str) -> dict[str, str]:\n    project = _gen_project(rng)\n    studio = _pick(rng, _STUDIOS)\n    role = _gen_role(rng)\n    desc = _gen_desc(rng)\n    deadline = _gen_deadline(rng)\n    sub_email = _gen_email(rng)\n    decoy_email = _pick(rng, _DECOY_EMAILS)\n    shoot_date = _pick(rng, _SHOOT_DATES)\n\n    if category == "standard_labeled":\n        header = rng.choice(["CASTING CALL", "AUDITION NOTICE", "URGENT REQUIREMENT", "OPEN CASTING"])\n        text = (\n            f"{header}\\n"\n            f"Project: {project}\\n"\n            f"Production: {studio}\\n"\n            f"Role: {role}\\n"\n            f"Profile: {desc}\\n"\n            f"Requirements: 30s slate + 2 min self-tape\\n"\n            f"Submission Deadline: {deadline}\\n"\n            f"Send profile & tape to: {sub_email}"\n        )\n        target = format_tagged_target(project, role, desc, deadline, sub_email)\n\n    elif category == "dual_email_trap":\n        text = (\n            f"CASTING BREAKDOWN — {project}\\n"\n            f"Agency Contact: {decoy_email} (general queries only)\\n"\n            f"Production House: {studio}\\n"\n            f"Looking for: {role}\\n"\n            f"Details: {desc}\\n"\n            f"Submit self-tape & headshots strictly to: {sub_email}\\n"\n            f"Last Date to Apply: {deadline}"\n        )\n        target = format_tagged_target(project, role, desc, deadline, sub_email)\n\n    elif category == "dual_date_trap":\n        text = (\n            f"{project} - UPCOMING FEATURE FILM\\n"\n            f"Tentative Shoot Dates: {shoot_date} in Mumbai\\n"\n            f"Character: {role} ({desc})\\n"\n            f"Audition Deadline: {deadline}\\n"\n            f"Mail entries to {sub_email}\\n"\n            f"Do not apply after {deadline}."\n        )\n        target = format_tagged_target(project, role, desc, deadline, sub_email)\n\n    elif category == "missing_fields":\n        omit_mode = rng.choice(["no_deadline", "no_email", "no_project", "no_deadline_no_email"])\n        if omit_mode == "no_deadline":\n            text = (\n                f"CASTING CALL - {project}\\n"\n                f"Role: {role}\\n"\n                f"Brief: {desc}\\n"\n                f"Send self-tape to {sub_email}"\n            )\n            target = format_tagged_target(project, role, desc, None, sub_email)\n        elif omit_mode == "no_email":\n            text = (\n                f"CASTING CALL – {project}\\n"\n                f"{studio} is looking for {role}. {desc}.\\n"\n                f"Audition Deadline: {deadline}. Shortlisted candidates will be contacted."\n            )\n            target = format_tagged_target(project, role, desc, deadline, None)\n        elif omit_mode == "no_project":\n            text = (\n                f"CASTING CALL – UPCOMING HINDI FEATURE FILM\\n"\n                f"{studio} is looking for {role}. {desc}. Freshers are welcome.\\n"\n                f"Audition Deadline: {deadline}\\n"\n                f"Email: {sub_email}"\n            )\n            target = format_tagged_target(None, role, desc, deadline, sub_email)\n        else:\n            text = (\n                f"REQUIREMENT FOR {project}\\n"\n                f"Seeking {role}: {desc}.\\n"\n                f"Send 3 recent pictures and intro video."\n            )\n            target = format_tagged_target(project, role, desc, None, None)\n\n    elif category == "whatsapp_prose":\n        city = rng.choice(["Mumbai", "Delhi", "Hyderabad", "London"])\n        text = (\n            f"Hi actors!! Casting for {project} ({studio}).\\n"\n            f"We need {role} — {desc}. Shoot in {city}.\\n"\n            f"Please share 2–3 recent headshots, intro clip, and audition video performing attached sides.\\n"\n            f"Kindly mail everything before {deadline} on {sub_email}. No fees!"\n        )\n        target = format_tagged_target(project, role, desc, deadline, sub_email)\n\n    elif category == "trailing_sides":\n        text = (\n            f"PROJECT: {project}\\n"\n            f"ROLE: {role}\\n"\n            f"NOTE: {desc}\\n"\n            f"DUE: {deadline}\\n"\n            f"SUBMIT: {sub_email}\\n\\n"\n            f"--- AUDITION SIDES ---\\n"\n            f"INT. CAFE - NIGHT (December 1, 1999)\\n"\n            f"DETECTIVE: Did you mail info@fake-dialogue.com yesterday?\\n"\n            f"PRIYA: No, I waited until Friday night."\n        )\n        target = format_tagged_target(project, role, desc, deadline, sub_email)\n\n    else:  # noisy_ocr (including multi-column flattening & symbols like –, ₹, |)\n        raw = (\n            f"CASTING CALL | {project}\\n"\n            f"{studio} | Role: {role}\\n"\n            f"{desc} | Compensation ₹25,000/day\\n"\n            f"Deadline: {deadline}\\n"\n            f"Email to Apply: {sub_email}"\n        )\n        text = _ocr_jitter(rng, raw)\n        target = format_tagged_target(project, role, desc, deadline, sub_email)\n\n    return {\n        "category": category,\n        "input": f"{PROMPT_PREFIX}{text}",\n        "target": target,\n    }\n\n\ndef generate_casting_corpus(\n    num_train: int = 18000,\n    num_val: int = 1000,\n    num_test: int = 1000,\n    seed: int = 42,\n) -> dict[str, list[dict[str, str]]]:\n    """Generates balanced train/val/test splits verified against Swift\'s grounding rule."""\n    rng = random.Random(seed)\n    categories = [\n        "standard_labeled",\n        "dual_email_trap",\n        "dual_date_trap",\n        "missing_fields",\n        "whatsapp_prose",\n        "trailing_sides",\n        "noisy_ocr",\n    ]\n\n    def build_split(count: int) -> list[dict[str, str]]:\n        rows: list[dict[str, str]] = []\n        for idx in range(count):\n            cat = categories[idx % len(categories)]\n            rows.append(_build_sample(rng, cat))\n        rng.shuffle(rows)\n        return rows\n\n    return {\n        "train": build_split(num_train),\n        "val": build_split(num_val),\n        "test": build_split(num_test),\n    }\n')
with open('/content/onetake_ml/vocab_surgery.py', 'w', encoding='utf-8') as f:
    f.write('"""Flan-T5 vocabulary surgery & pure-Python SentencePiece Unigram Viterbi reference.\n\nBoth Python training/evaluation and Swift `UnigramTokenizer` share the exact `sp_vocab.json`\ntable and preserve newlines (`\\\\n`) as `<0x0A>` so multi-line OCR layouts and tagged output\nlines round-trip losslessly with 100% token-ID parity.\n"""\n\nfrom __future__ import annotations\n\nimport json\nimport math\nimport re\nfrom collections import Counter\nfrom typing import Iterable\n\n\n_BYTE_TOKENS = [f"<0x{b:02X}>" for b in range(256)]\n_DOMAIN_ESSENTIALS = [\n    "\\u2581",\n    ":",\n    "-",\n    "/",\n    ".",\n    ",",\n    "@",\n    "(",\n    ")",\n    "|",\n    "&",\n    "!",\n    "?",\n    "%",\n    "\'",\n    \'"\',\n    "\\u2581PROJECT",\n    "\\u2581ROLE",\n    "\\u2581DESC",\n    "\\u2581DEADLINE",\n    "\\u2581EMAIL",\n    "\\u2581NONE",\n    "PROJECT",\n    "ROLE",\n    "DESC",\n    "DEADLINE",\n    "EMAIL",\n    "NONE",\n    "parse",\n    "\\u2581parse",\n    "\\u2581audition",\n    "\\u2581notice",\n]\n\n\n_SYMBOL_REPLACEMENTS = (\n    ("\\r\\n", "\\n"),\n    ("\\r", "\\n"),\n    ("\\u2013", "-"),   # en dash\n    ("\\u2014", "-"),   # em dash\n    ("\\u2212", "-"),   # minus sign\n    ("\\u2018", "\'"),   # left single quote\n    ("\\u2019", "\'"),   # right single quote\n    ("\\u201c", \'"\'),   # left double quote\n    ("\\u201d", \'"\'),   # right double quote\n    ("\\u2026", "..."), # ellipsis\n    ("\\u20b9", "Rs."), # rupee symbol\n    ("\\u00a0", " "),   # nbsp\n)\n\n\ndef _normalize_input_symbols(text: str) -> str:\n    """Normalizes common casting-flyer punctuation missing from T5\'s non-byte vocab."""\n    for src, dst in _SYMBOL_REPLACEMENTS:\n        if src in text:\n            text = text.replace(src, dst)\n    return text\n\n\ndef _split_preserving_newlines(text: str) -> list[tuple[str, bool]]:\n    """Splits text into chunks, returning `(chunk, is_newline)` pairs."""\n    text = _normalize_input_symbols(text)\n    parts: list[tuple[str, bool]] = []\n    for idx, line in enumerate(text.split("\\n")):\n        if idx > 0:\n            parts.append(("\\n", True))\n        if line:\n            parts.append((line, False))\n    return parts\n\n\ndef _normalize_line_segment(segment: str) -> str:\n    """Normalizes spaces to SentencePiece `▁` with dummy leading `▁` (exact Swift mirror)."""\n    collapsed = re.sub(r"[ \\t\\r]+", " ", segment).strip(" ")\n    if not collapsed:\n        return ""\n    return "\\u2581" + collapsed.replace(" ", "\\u2581")\n\n\ndef unigram_encode_reference(\n    text: str,\n    vocab: list[list],\n    append_eos: bool = True,\n) -> list[int]:\n    """Viterbi DP SentencePiece Unigram encoder matching Swift `UnigramTokenizer`."""\n    piece_to_id: dict[str, int] = {}\n    piece_to_score: dict[str, float] = {}\n    max_piece_len = 1\n    for idx, entry in enumerate(vocab):\n        piece, score = entry[0], float(entry[1])\n        piece_to_id[piece] = idx\n        piece_to_score[piece] = score\n        if not (piece.startswith("<") and piece.endswith(">")):\n            if len(piece) > max_piece_len:\n                max_piece_len = len(piece)\n\n    eos_id = piece_to_id.get("</s>", 1)\n    unk_id = piece_to_id.get("<unk>", 2)\n    newline_id = piece_to_id.get("<0x0A>", piece_to_id.get("<extra_id_0>", unk_id))\n    ids: list[int] = []\n\n    for segment, is_newline in _split_preserving_newlines(text):\n        if is_newline:\n            ids.append(newline_id)\n            continue\n\n        norm = _normalize_line_segment(segment)\n        if not norm:\n            continue\n\n        n = len(norm)\n        best_score = [-math.inf] * (n + 1)\n        best_len = [0] * (n + 1)\n        best_score[0] = 0.0\n\n        for i in range(n):\n            if best_score[i] == -math.inf:\n                continue\n            limit = min(n, i + max_piece_len)\n            matched_any = False\n            for j in range(i + 1, limit + 1):\n                sub = norm[i:j]\n                if sub in piece_to_score:\n                    cand = best_score[i] + piece_to_score[sub]\n                    if cand > best_score[j]:\n                        best_score[j] = cand\n                        best_len[j] = j - i\n                        matched_any = True\n            # Single-character byte-fallback transition when no piece covers norm[i]\n            if best_len[i + 1] == 0:\n                fallback_penalty = -30.0\n                cand = best_score[i] + fallback_penalty\n                if cand > best_score[i + 1]:\n                    best_score[i + 1] = cand\n                    best_len[i + 1] = -1  # negative indicates byte fallback of norm[i]\n            elif not matched_any and best_score[i + 1] == -math.inf:\n                best_score[i + 1] = best_score[i] - 30.0\n                best_len[i + 1] = -1\n\n        # Backtrack\n        spans: list[tuple[int, int]] = []\n        pos = n\n        while pos > 0:\n            step = best_len[pos]\n            if step > 0:\n                start = pos - step\n                spans.append((start, pos))\n                pos = start\n            else:\n                spans.append((pos - 1, -1))\n                pos -= 1\n        spans.reverse()\n\n        for start, end in spans:\n            if end == -1:\n                ch = norm[start]\n                for b in ch.encode("utf-8"):\n                    ids.append(piece_to_id.get(f"<0x{b:02X}>", unk_id))\n            else:\n                ids.append(piece_to_id.get(norm[start:end], unk_id))\n\n    if append_eos:\n        ids.append(eos_id)\n    return ids\n\n\ndef unigram_decode_reference(\n    ids: Iterable[int],\n    vocab: list[list],\n    skip_ids: set[int] = {0, 1, 2},\n) -> str:\n    """Decodes Unigram IDs including UTF-8 `<0xXX>` byte sequences back to `str`."""\n    parts: list[str] = []\n    byte_buf = bytearray()\n\n    def flush_bytes() -> None:\n        if byte_buf:\n            parts.append(byte_buf.decode("utf-8", errors="replace"))\n            byte_buf.clear()\n\n    vocab_len = len(vocab)\n    newline_id = next(\n        (idx for idx, entry in enumerate(vocab) if entry[0] == "<0x0A>"),\n        next((idx for idx, entry in enumerate(vocab) if entry[0] == "<extra_id_0>"), None),\n    )\n    for idx in ids:\n        if idx in skip_ids or idx < 0 or idx >= vocab_len:\n            continue\n        if newline_id is not None and idx == newline_id:\n            flush_bytes()\n            parts.append("\\n")\n            continue\n        piece = vocab[idx][0]\n        if len(piece) == 6 and piece.startswith("<0x") and piece.endswith(">"):\n            try:\n                byte_buf.append(int(piece[3:5], 16))\n                continue\n            except ValueError:\n                pass\n        flush_bytes()\n        parts.append(piece.replace("\\u2581", " "))\n\n    flush_bytes()\n    text = "".join(parts)\n    if text.startswith(" "):\n        text = text[1:]\n    # Strip dummy space injected after newline when encoding multi-line text\n    return text.replace("\\n ", "\\n")\n\n\ndef build_pruned_piece_table(\n    pieces_with_scores: list[tuple[str, float]],\n    corpus_texts: list[str],\n    target_vocab_size: int = 12000,\n) -> dict:\n    """Selects up to `target_vocab_size` pieces preserving specials, 256 bytes, & domain tokens."""\n    total_src = len(pieces_with_scores)\n    target_vocab_size = min(target_vocab_size, total_src)\n    piece_to_old_id = {p: idx for idx, (p, _) in enumerate(pieces_with_scores)}\n\n    # 1. Mandatory IDs 0, 1, 2\n    selected_old_ids: list[int] = [0, 1, 2]\n    selected_set = set(selected_old_ids)\n\n    def add_piece(p: str) -> None:\n        old_id = piece_to_old_id.get(p)\n        if old_id is not None and old_id not in selected_set and len(selected_old_ids) < target_vocab_size:\n            selected_set.add(old_id)\n            selected_old_ids.append(old_id)\n\n    # 2. Dedicated newline token (`<0x0A>` if byte-fallback exists, else `<extra_id_0>`) + byte fallback tokens\n    if "<0x0A>" not in piece_to_old_id:\n        add_piece("<extra_id_0>")\n    for bt in _BYTE_TOKENS:\n        add_piece(bt)\n\n    # 3. Essential ASCII & domain tag tokens\n    for tok in _DOMAIN_ESSENTIALS:\n        add_piece(tok)\n    for code in range(33, 127):\n        ch = chr(code)\n        add_piece(ch)\n        add_piece("\\u2581" + ch)\n\n    # 4. Frequency count via unigram encoding over original full vocab\n    full_vocab = [[p, float(s)] for p, s in pieces_with_scores]\n    usage_counts: Counter[int] = Counter()\n    for text in corpus_texts:\n        for tid in unigram_encode_reference(text, full_vocab, append_eos=False):\n            usage_counts[tid] += 1\n\n    # Rank remaining pieces: corpus hits first, then higher Unigram log-prob (and Latin preference)\n    remaining: list[tuple[float, int]] = []\n    for old_id, (piece, score) in enumerate(pieces_with_scores):\n        if old_id in selected_set:\n            continue\n        if piece.startswith("<extra_id_"):\n            continue\n        hits = usage_counts.get(old_id, 0)\n        latin_bonus = 2.0 if piece.replace("\\u2581", "").isascii() else 0.0\n        rank_key = hits * 1000.0 + float(score) + latin_bonus\n        remaining.append((rank_key, old_id))\n\n    remaining.sort(key=lambda item: item[0], reverse=True)\n    for _, old_id in remaining:\n        if len(selected_old_ids) >= target_vocab_size:\n            break\n        selected_set.add(old_id)\n        selected_old_ids.append(old_id)\n\n    pruned_vocab = [[pieces_with_scores[oid][0], float(pieces_with_scores[oid][1])] for oid in selected_old_ids]\n    return {\n        "old_to_new_ids": selected_old_ids,\n        "vocab": pruned_vocab,\n        "pad_id": 0,\n        "eos_id": 1,\n        "unk_id": 2,\n    }\n\n\ndef save_sp_vocab_json(pruned: dict, path: str) -> None:\n    with open(path, "w", encoding="utf-8") as f:\n        json.dump(pruned["vocab"], f, ensure_ascii=False, separators=(",", ":"))\n')
with open('/content/onetake_ml/train_and_qat.py', 'w', encoding='utf-8') as f:
    f.write('"""Flan-T5-Small vocabulary surgery, full fine-tuning, and 4-bit blockwise QAT."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport math\nimport os\nfrom typing import Any\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader, Dataset\nfrom transformers import AutoModelForSeq2SeqLM, AutoTokenizer, get_cosine_schedule_with_warmup\n\nfrom dataset_generator import generate_casting_corpus\nfrom vocab_surgery import (\n    _normalize_input_symbols,\n    build_pruned_piece_table,\n    save_sp_vocab_json,\n    unigram_decode_reference,\n    unigram_encode_reference,\n)\n\n\ndef extract_sentencepiece_table(tokenizer: Any, model_name: str = "google/flan-t5-small") -> list[tuple[str, float]]:\n    """Extracts `(piece, score)` pairs from T5 SentencePiece plus sentinel tokens (up to 32100)."""\n    sp = getattr(tokenizer, "sp_model", None)\n    if sp is None:\n        import sentencepiece as spm\n        from huggingface_hub import hf_hub_download\n\n        sp_file = getattr(tokenizer, "vocab_file", None) or hf_hub_download(model_name, "spiece.model", local_files_only=True)\n        sp = spm.SentencePieceProcessor(model_file=sp_file)\n\n    pieces: list[tuple[str, float]] = [\n        (sp.id_to_piece(i), float(sp.get_score(i))) for i in range(sp.get_piece_size())\n    ]\n    # Append HF T5 added sentinel tokens (<extra_id_99> .. <extra_id_0> at indices 32000..32099)\n    total_vocab = max(len(pieces), int(getattr(tokenizer, "vocab_size", len(pieces))))\n    if len(pieces) < total_vocab:\n        for idx in range(len(pieces), total_vocab):\n            tok = tokenizer.convert_ids_to_tokens(idx)\n            pieces.append((tok if isinstance(tok, str) else f"<extra_id_{total_vocab - 1 - idx}>", -10.0))\n    return pieces\n\n\ndef prune_flan_t5_vocab(\n    model: nn.Module,\n    old_to_new_ids: list[int],\n) -> nn.Module:\n    """Slices T5 embedding and LM head weights to the pruned vocabulary indices."""\n    new_vocab_size = len(old_to_new_ids)\n    idx_tensor = torch.tensor(old_to_new_ids, dtype=torch.long)\n\n    with torch.no_grad():\n        old_shared = model.shared.weight.data[idx_tensor].clone()\n        old_lm_head = model.lm_head.weight.data[idx_tensor].clone()\n\n        new_shared = nn.Embedding(new_vocab_size, model.config.d_model)\n        new_shared.weight.data.copy_(old_shared)\n        model.shared = new_shared\n        model.encoder.set_input_embeddings(new_shared)\n        model.decoder.set_input_embeddings(new_shared)\n\n        new_lm_head = nn.Linear(model.config.d_model, new_vocab_size, bias=False)\n        new_lm_head.weight.data.copy_(old_lm_head)\n        model.lm_head = new_lm_head\n\n    model.config.vocab_size = new_vocab_size\n    model.config.pad_token_id = 0\n    model.config.eos_token_id = 1\n    model.config.decoder_start_token_id = 0\n    return model\n\n\nclass NoticeSeq2SeqDataset(Dataset):\n    def __init__(\n        self,\n        rows: list[dict[str, str]],\n        vocab: list[list],\n        max_source_len: int = 512,\n        max_target_len: int = 96,\n    ) -> None:\n        self.samples: list[tuple[list[int], list[int]]] = []\n        for row in rows:\n            src_ids = unigram_encode_reference(row["input"], vocab, append_eos=True)[:max_source_len]\n            if src_ids[-1] != 1:\n                src_ids[-1] = 1\n            tgt_ids = unigram_encode_reference(row["target"], vocab, append_eos=True)[:max_target_len]\n            if tgt_ids[-1] != 1:\n                tgt_ids[-1] = 1\n            self.samples.append((src_ids, tgt_ids))\n\n    def __len__(self) -> int:\n        return len(self.samples)\n\n    def __getitem__(self, idx: int) -> tuple[list[int], list[int]]:\n        return self.samples[idx]\n\n\ndef collate_seq2seq(batch: list[tuple[list[int], list[int]]]) -> dict[str, torch.Tensor]:\n    max_src = max(len(x[0]) for x in batch)\n    max_tgt = max(len(x[1]) for x in batch)\n    bsz = len(batch)\n\n    input_ids = torch.zeros((bsz, max_src), dtype=torch.long)\n    attention_mask = torch.zeros((bsz, max_src), dtype=torch.long)\n    labels = torch.full((bsz, max_tgt), -100, dtype=torch.long)\n\n    for i, (src, tgt) in enumerate(batch):\n        input_ids[i, : len(src)] = torch.tensor(src, dtype=torch.long)\n        attention_mask[i, : len(src)] = 1\n        labels[i, : len(tgt)] = torch.tensor(tgt, dtype=torch.long)\n\n    return {\n        "input_ids": input_ids,\n        "attention_mask": attention_mask,\n        "labels": labels,\n    }\n\n\ndef _fake_quant_int4_blockwise(w: torch.Tensor, block_size: int = 32) -> torch.Tensor:\n    """Straight-through estimator simulating symmetric 4-bit per-block quantization."""\n    orig_shape = w.shape\n    flat = w.reshape(-1, orig_shape[-1])\n    pad_cols = (block_size - (flat.shape[1] % block_size)) % block_size\n    if pad_cols:\n        flat_padded = F.pad(flat, (0, pad_cols))\n    else:\n        flat_padded = flat\n    blocks = flat_padded.reshape(-1, block_size)\n    max_abs = blocks.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6)\n    scale = max_abs / 7.0\n    q = torch.clamp(torch.round(blocks / scale), -8, 7)\n    deq = q * scale\n    if pad_cols:\n        deq = deq.reshape(flat.shape[0], -1)[:, : flat.shape[1]]\n    return w + (deq.reshape(orig_shape) - w).detach()\n\n\nclass QATLinear(nn.Module):\n    def __init__(self, linear: nn.Linear, block_size: int = 32) -> None:\n        super().__init__()\n        self.linear = linear\n        self.block_size = block_size\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        w_q = _fake_quant_int4_blockwise(self.linear.weight, self.block_size)\n        return F.linear(x, w_q, self.linear.bias)\n\n\ndef wrap_linear_modules_for_qat(module: nn.Module, block_size: int = 32) -> nn.Module:\n    for name, child in list(module.named_children()):\n        if isinstance(child, nn.Linear):\n            setattr(module, name, QATLinear(child, block_size=block_size))\n        else:\n            wrap_linear_modules_for_qat(child, block_size=block_size)\n    return module\n\n\ndef unwrap_qat_modules(module: nn.Module) -> nn.Module:\n    for name, child in list(module.named_children()):\n        if isinstance(child, QATLinear):\n            setattr(module, name, child.linear)\n        else:\n            unwrap_qat_modules(child)\n    return module\n\n\ndef parse_tagged_dict(raw: str) -> dict[str, str]:\n    parsed: dict[str, str] = {}\n    for line in raw.splitlines():\n        if ":" not in line:\n            continue\n        k, v = line.split(":", 1)\n        parsed[k.strip().upper()] = v.strip()\n    return parsed\n\n\n@torch.no_grad()\ndef evaluate_validation(\n    model: nn.Module,\n    val_rows: list[dict[str, str]],\n    vocab: list[list],\n    device: torch.device,\n    max_samples: int = 250,\n) -> dict[str, float]:\n    model.eval()\n    subset = val_rows[:max_samples]\n    keys = ["PROJECT", "ROLE", "DESC", "DEADLINE", "EMAIL"]\n    exact_hits = {k: 0 for k in keys}\n    none_total = 0\n    none_correct = 0\n\n    for idx, row in enumerate(subset):\n        src_ids = unigram_encode_reference(row["input"], vocab, append_eos=True)[:512]\n        inp = torch.tensor([src_ids], dtype=torch.long, device=device)\n        gen = model.generate(input_ids=inp, max_new_tokens=96, do_sample=False, eos_token_id=1, pad_token_id=0)\n        pred_text = unigram_decode_reference(gen[0].tolist(), vocab)\n        pred_map = parse_tagged_dict(pred_text)\n        gold_map = parse_tagged_dict(_normalize_input_symbols(row["target"]))\n        if idx < 2:\n            print(f"  [Sample {idx}] PRED={pred_text!r} | GOLD={row[\'target\']!r}")\n        for k in keys:\n            g = gold_map.get(k, "NONE")\n            p = pred_map.get(k, "NONE")\n            if p.lower() == g.lower():\n                exact_hits[k] += 1\n            if g == "NONE":\n                none_total += 1\n                if p.upper() == "NONE":\n                    none_correct += 1\n\n    metrics = {f"{k.lower()}_acc": exact_hits[k] / max(1, len(subset)) for k in keys}\n    metrics["mean_field_acc"] = sum(metrics.values()) / len(keys)\n    metrics["none_abstention_acc"] = none_correct / max(1, none_total)\n    return metrics\n\n\ndef train_pipeline(\n    output_dir: str,\n    base_model_name: str = "google/flan-t5-small",\n    vocab_size: int = 8192,\n    num_train: int = 18000,\n    num_val: int = 1000,\n    epochs: int = 5,\n    qat_epochs: int = 1,\n    batch_size: int = 32,\n    lr: float = 3e-4,\n    qat_lr: float = 5e-5,\n) -> dict[str, Any]:\n    os.makedirs(output_dir, exist_ok=True)\n    device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))\n    use_bf16 = device.type == "cuda" and torch.cuda.is_bf16_supported()\n\n    print(f"[OneTake] Device={device}, bf16={use_bf16}, vocab_size={vocab_size}")\n    corpus = generate_casting_corpus(num_train=num_train, num_val=num_val, num_test=500, seed=42)\n\n    hf_tok = AutoTokenizer.from_pretrained(base_model_name, use_fast=False)\n    pieces_with_scores = extract_sentencepiece_table(hf_tok)\n    corpus_texts = [r["input"] for r in corpus["train"]] + [r["target"] for r in corpus["train"]]\n    pruned = build_pruned_piece_table(pieces_with_scores, corpus_texts, target_vocab_size=vocab_size)\n    vocab = pruned["vocab"]\n    save_sp_vocab_json(pruned, os.path.join(output_dir, "sp_vocab.json"))\n\n    # Save 20 golden parity vectors for Swift `UnigramTokenizer` tests\n    golden_samples = [r["input"] for r in corpus["test"][:15]] + [\n        "CASTING CALL – UPCOMING HINDI FEATURE FILM\\nRole: Female Lead ₹25,000/day\\nEmail: abhinav@mcc.in",\n        "PROJECT: MCCC - KOSHISHE\\nROLE: Male Lead\\nDEADLINE: 15 December, 2026\\nEMAIL: abhinav@mcc.in",\n        "Line with emoji 🎬 & Devanagari हिन्दी & curly quotes “Action!”",\n        "PHOTON — Casting Call\\nMen aged 22–25 years\\nDeadline: 30 November 2026",\n        "No deadline notice\\twith tabs   and multiple spaces.",\n    ]\n    golden_vectors = [\n        {"text": s, "ids": unigram_encode_reference(s, vocab, append_eos=True)}\n        for s in golden_samples\n    ]\n    with open(os.path.join(output_dir, "golden_parity_vectors.json"), "w", encoding="utf-8") as f:\n        json.dump(golden_vectors, f, ensure_ascii=False, indent=2)\n\n    model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)\n    model = prune_flan_t5_vocab(model, pruned["old_to_new_ids"]).to(device)\n\n    train_ds = NoticeSeq2SeqDataset(corpus["train"], vocab)\n    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_seq2seq)\n\n    def run_epochs(n_epochs: int, learning_rate: float, label_smoothing: float, phase: str) -> None:\n        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)\n        total_steps = max(1, len(train_loader) * n_epochs)\n        scheduler = get_cosine_schedule_with_warmup(optimizer, int(0.06 * total_steps), total_steps)\n        loss_fn = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=label_smoothing)\n\n        for epoch in range(1, n_epochs + 1):\n            model.train()\n            running = 0.0\n            for step, batch in enumerate(train_loader, start=1):\n                optimizer.zero_grad(set_to_none=True)\n                inp = batch["input_ids"].to(device)\n                mask = batch["attention_mask"].to(device)\n                lbl = batch["labels"].to(device)\n                with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):\n                    out = model(input_ids=inp, attention_mask=mask, labels=lbl)\n                    loss = loss_fn(out.logits.view(-1, out.logits.size(-1)), lbl.view(-1))\n                loss.backward()\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                optimizer.step()\n                scheduler.step()\n                running += float(loss.item())\n                if step % 150 == 0:\n                    print(f"[{phase}] epoch {epoch}/{n_epochs} step {step}/{len(train_loader)} loss={running/step:.4f}")\n            metrics = evaluate_validation(model, corpus["val"], vocab, device)\n            print(f"[{phase}] epoch {epoch} val metrics: {metrics}")\n\n    run_epochs(epochs, lr, label_smoothing=0.05, phase="Stage1-BF16")\n    if qat_epochs > 0:\n        wrap_linear_modules_for_qat(model, block_size=32)\n        run_epochs(qat_epochs, qat_lr, label_smoothing=0.0, phase="Stage2-INT4-QAT")\n        unwrap_qat_modules(model)\n\n    final_metrics = evaluate_validation(model, corpus["test"], vocab, device, max_samples=500)\n    print(f"[Final Held-Out Test Metrics] {final_metrics}")\n\n    ckpt_dir = os.path.join(output_dir, "hf_checkpoint")\n    model.cpu().save_pretrained(ckpt_dir)\n    with open(os.path.join(output_dir, "eval_metrics.json"), "w", encoding="utf-8") as f:\n        json.dump(final_metrics, f, indent=2)\n    return final_metrics\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--output-dir", default="artifacts/flan_t5_onetake")\n    parser.add_argument("--vocab-size", type=int, default=8192)\n    parser.add_argument("--num-train", type=int, default=18000)\n    parser.add_argument("--epochs", type=int, default=5)\n    parser.add_argument("--qat-epochs", type=int, default=1)\n    parser.add_argument("--batch-size", type=int, default=32)\n    args = parser.parse_args()\n    train_pipeline(\n        output_dir=args.output_dir,\n        vocab_size=args.vocab_size,\n        num_train=args.num_train,\n        epochs=args.epochs,\n        qat_epochs=args.qat_epochs,\n        batch_size=args.batch_size,\n    )\n')
with open('/content/onetake_ml/export_coreml.py', 'w', encoding='utf-8') as f:
    f.write('"""Core ML encoder/decoder exporter with 4-bit blockwise quantization and 35 MB size gate."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport math\nimport os\nimport shutil\nfrom typing import Any\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef _compute_relative_position_bucket(\n    relative_position: torch.Tensor,\n    bidirectional: bool = True,\n    num_buckets: int = 32,\n    max_distance: int = 128,\n) -> torch.Tensor:\n    ret = torch.zeros_like(relative_position, dtype=torch.long)\n    n = -relative_position\n    if bidirectional:\n        num_buckets //= 2\n        ret += (n < 0).to(torch.long) * num_buckets\n        n = torch.abs(n)\n    else:\n        n = torch.maximum(n, torch.zeros_like(n))\n    max_exact = num_buckets // 2\n    is_small = n < max_exact\n    val_if_large = max_exact + (\n        torch.log(n.float().clamp(min=1e-6) / max_exact)\n        / math.log(max_distance / max_exact)\n        * (num_buckets - max_exact)\n    ).to(torch.long)\n    val_if_large = torch.minimum(val_if_large, torch.full_like(val_if_large, num_buckets - 1))\n    return ret + torch.where(is_small, n.to(torch.long), val_if_large)\n\n\ndef _t5_attention_forward(\n    attn_mod: nn.Module,\n    hidden_states: torch.Tensor,\n    key_value_states: torch.Tensor | None,\n    mask: torch.Tensor | None,\n    position_bias: torch.Tensor | None,\n    rel_buckets: torch.Tensor | None,\n) -> tuple[torch.Tensor, torch.Tensor | None]:\n    n_heads = int(attn_mod.n_heads)\n    d_kv = int(attn_mod.key_value_proj_dim)\n\n    q = attn_mod.q(hidden_states).reshape(1, -1, n_heads, d_kv).transpose(1, 2)\n    kv_src = hidden_states if key_value_states is None else key_value_states\n    k = attn_mod.k(kv_src).reshape(1, -1, n_heads, d_kv).transpose(1, 2)\n    v = attn_mod.v(kv_src).reshape(1, -1, n_heads, d_kv).transpose(1, 2)\n\n    scores = torch.matmul(q, k.transpose(2, 3))\n    if position_bias is None and attn_mod.has_relative_attention_bias and rel_buckets is not None:\n        bias_vals = attn_mod.relative_attention_bias(rel_buckets)  # (q_len, k_len, H)\n        position_bias = bias_vals.permute(2, 0, 1).unsqueeze(0)   # (1, H, q_len, k_len)\n\n    if position_bias is not None:\n        scores = scores + position_bias\n    if mask is not None:\n        scores = scores + mask\n\n    probs = F.softmax(scores.float(), dim=-1).to(scores.dtype)\n    ctx = torch.matmul(probs, v).transpose(1, 2).reshape(1, -1, n_heads * d_kv)\n    return attn_mod.o(ctx), position_bias\n\n\nclass T5EncoderWrapper(nn.Module):\n    """Pure-op T5 encoder wrapper avoiding HF dynamic masking ops for Core ML tracing."""\n\n    def __init__(self, t5_model: Any) -> None:\n        super().__init__()\n        self.embed = t5_model.shared\n        self.blocks = t5_model.encoder.block\n        self.final_ln = t5_model.encoder.final_layer_norm\n        pos = torch.arange(512).unsqueeze(0)\n        rel = pos - pos.transpose(0, 1)\n        self.register_buffer("rel_buckets", _compute_relative_position_bucket(rel, bidirectional=True).to(torch.int32))\n\n    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:\n        x = self.embed(input_ids.to(torch.long))\n        s = input_ids.size(1)\n        enc_mask = (1.0 - attention_mask[:, None, None, :].float()) * (-1e9)\n        rel_b = self.rel_buckets[:s, :s].to(torch.long)\n        pos_bias: torch.Tensor | None = None\n        for block in self.blocks:\n            normed = block.layer[0].layer_norm(x)\n            attn_out, pos_bias = _t5_attention_forward(\n                block.layer[0].SelfAttention, normed, None, enc_mask, pos_bias, rel_b\n            )\n            x = x + attn_out\n            x = block.layer[1](x)\n        return self.final_ln(x)\n\n\nclass T5DecoderWrapper(nn.Module):\n    """Pure-op T5 decoder + LM head wrapper without HF dynamic cache or new_ones ops."""\n\n    def __init__(self, t5_model: Any) -> None:\n        super().__init__()\n        self.embed = t5_model.shared\n        self.blocks = t5_model.decoder.block\n        self.final_ln = t5_model.decoder.final_layer_norm\n        self.lm_head = t5_model.lm_head\n        self.model_dim = float(t5_model.config.d_model)\n        # Only scale when embeddings are actually weight-tied (Flan-T5-Small stores untied lm_head)\n        self.tie_word_embeddings = bool(\n            getattr(t5_model.config, "tie_word_embeddings", False)\n            and t5_model.shared.weight.data_ptr() == t5_model.lm_head.weight.data_ptr()\n        )\n        pos = torch.arange(96).unsqueeze(0)\n        rel = pos - pos.transpose(0, 1)\n        self.register_buffer("rel_buckets", _compute_relative_position_bucket(rel, bidirectional=False).to(torch.int32))\n        causal = torch.triu(torch.full((1, 1, 96, 96), -1e9, dtype=torch.float32), diagonal=1)\n        self.register_buffer("causal_mask", causal)\n\n    def forward(\n        self,\n        decoder_input_ids: torch.Tensor,\n        encoder_hidden_states: torch.Tensor,\n        encoder_attention_mask: torch.Tensor,\n    ) -> torch.Tensor:\n        x = self.embed(decoder_input_ids.to(torch.long))\n        k = decoder_input_ids.size(1)\n        self_mask = self.causal_mask[:, :, :k, :k]\n        enc_mask = (1.0 - encoder_attention_mask[:, None, None, :].float()) * (-1e9)\n        rel_b = self.rel_buckets[:k, :k].to(torch.long)\n        pos_bias: torch.Tensor | None = None\n\n        for block in self.blocks:\n            normed = block.layer[0].layer_norm(x)\n            self_out, pos_bias = _t5_attention_forward(\n                block.layer[0].SelfAttention, normed, None, self_mask, pos_bias, rel_b\n            )\n            x = x + self_out\n            cross_normed = block.layer[1].layer_norm(x)\n            cross_out, _ = _t5_attention_forward(\n                block.layer[1].EncDecAttention, cross_normed, encoder_hidden_states, enc_mask, None, None\n            )\n            x = x + cross_out\n            x = block.layer[2](x)\n\n        seq_out = self.final_ln(x)\n        if self.tie_word_embeddings:\n            seq_out = seq_out * (self.model_dim ** -0.5)\n        return self.lm_head(seq_out)\n\n\ndef _dir_size_bytes(path: str) -> int:\n    total = 0\n    for root, _, files in os.walk(path):\n        for fname in files:\n            total += os.path.getsize(os.path.join(root, fname))\n    return total\n\n\ndef _quantize_4bit(mlmodel: Any, vocab_size: int | None = None) -> Any:\n    """Compresses transformer blocks to INT4 (block_size=64) and embeddings/lm_head to 6-bit k-means."""\n    import coremltools.optimize.coreml as cto\n\n    lm_ops: list[str] = []\n    if vocab_size is not None:\n        main_fn = mlmodel._mil_program.functions.get("main")\n        if main_fn is not None:\n            lm_ops = [\n                op.name\n                for op in main_fn.operations\n                if op.op_type == "linear" and len(op.outputs[0].shape) >= 1 and op.outputs[0].shape[-1] == vocab_size\n            ]\n\n    q4_cfg = cto.OpLinearQuantizerConfig(\n        mode="linear_symmetric",\n        dtype="int4",\n        granularity="per_block",\n        block_size=64,\n        weight_threshold=512,\n    )\n    q4_opt = cto.OptimizationConfig(\n        global_config=q4_cfg,\n        op_type_configs={"gather": None},\n        op_name_configs={name: None for name in lm_ops},\n    )\n    compressed = cto.linear_quantize_weights(mlmodel, config=q4_opt)\n\n    try:\n        pal6 = cto.OpPalettizerConfig(\n            mode="kmeans",\n            nbits=6,\n            granularity="per_grouped_channel",\n            group_size=64,\n            weight_threshold=512,\n        )\n        p6_opt = cto.OptimizationConfig(\n            global_config=None,\n            op_type_configs={"gather": pal6},\n            op_name_configs={name: pal6 for name in lm_ops},\n        )\n        return cto.palettize_weights(compressed, config=p6_opt)\n    except Exception:\n        pal6_uniform = cto.OpPalettizerConfig(\n            mode="uniform",\n            nbits=6,\n            granularity="per_grouped_channel",\n            group_size=64,\n            weight_threshold=512,\n        )\n        p6_opt = cto.OptimizationConfig(\n            global_config=None,\n            op_type_configs={"gather": pal6_uniform},\n            op_name_configs={name: pal6_uniform for name in lm_ops},\n        )\n        return cto.palettize_weights(compressed, config=p6_opt)\n\n\ndef _stamp_metadata(mlmodel: Any, author: str, description: str) -> None:\n    mlmodel.author = author\n    mlmodel.license = "OneTake Proprietary"\n    mlmodel.short_description = description\n    mlmodel.version = "2.0.0"\n\n\ndef export_coreml_models(\n    checkpoint_dir: str,\n    output_dir: str,\n    sp_vocab_path: str | None = None,\n    author: str = "Porage-Lover <jitund05@gmail.com>",\n    max_total_mb: float = 35.0,\n) -> dict[str, Any]:\n    """Traces, converts, and 4-bit quantizes pruned Flan-T5-Small to Core ML."""\n    import coremltools as ct\n    from transformers import AutoModelForSeq2SeqLM\n\n    os.makedirs(output_dir, exist_ok=True)\n    model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_dir).eval().cpu()\n    d_model = int(model.config.d_model)\n\n    enc_wrap = T5EncoderWrapper(model).eval()\n    dec_wrap = T5DecoderWrapper(model).eval()\n\n    ex_enc_ids = torch.ones((1, 256), dtype=torch.int32)\n    ex_enc_mask = torch.ones((1, 256), dtype=torch.int32)\n    ex_hidden = enc_wrap(ex_enc_ids, ex_enc_mask)\n    ex_dec_ids = torch.zeros((1, 16), dtype=torch.int32)\n\n    with torch.no_grad():\n        traced_enc = torch.jit.trace(enc_wrap, (ex_enc_ids, ex_enc_mask))\n        traced_dec = torch.jit.trace(dec_wrap, (ex_dec_ids, ex_hidden, ex_enc_mask))\n        check_hidden = traced_enc(ex_enc_ids, ex_enc_mask)\n        check_logits = traced_dec(ex_dec_ids, check_hidden, ex_enc_mask)\n        assert torch.isfinite(check_hidden).all(), "Encoder trace produced non-finite values"\n        assert torch.isfinite(check_logits).all(), "Decoder trace produced non-finite values"\n\n    s_dim = ct.RangeDim(lower_bound=1, upper_bound=512, default=256)\n    k_dim = ct.RangeDim(lower_bound=1, upper_bound=96, default=16)\n\n    # Use FLOAT32 compute precision so T5 attention masking (-3.4e38) and GeGLU/LayerNorm never overflow FP16 to NaN.\n    # Weights remain 4-bit blockwise compressed on disk to fit the <= 35 MB bundle cap.\n    enc_ml = ct.convert(\n        traced_enc,\n        source="pytorch",\n        convert_to="mlprogram",\n        minimum_deployment_target=ct.target.iOS18,\n        compute_precision=ct.precision.FLOAT32,\n        inputs=[\n            ct.TensorType(name="input_ids", shape=(1, s_dim), dtype=int),\n            ct.TensorType(name="attention_mask", shape=(1, s_dim), dtype=int),\n        ],\n        outputs=[ct.TensorType(name="encoder_hidden_states")],\n    )\n\n    dec_ml = ct.convert(\n        traced_dec,\n        source="pytorch",\n        convert_to="mlprogram",\n        minimum_deployment_target=ct.target.iOS18,\n        compute_precision=ct.precision.FLOAT32,\n        inputs=[\n            ct.TensorType(name="decoder_input_ids", shape=(1, k_dim), dtype=int),\n            ct.TensorType(name="encoder_hidden_states", shape=(1, s_dim, d_model)),\n            ct.TensorType(name="encoder_attention_mask", shape=(1, s_dim), dtype=int),\n        ],\n        outputs=[ct.TensorType(name="logits")],\n    )\n\n    vocab_size = int(model.config.vocab_size)\n    enc_q = _quantize_4bit(enc_ml, vocab_size=vocab_size)\n    dec_q = _quantize_4bit(dec_ml, vocab_size=vocab_size)\n    _stamp_metadata(enc_q, author, "OneTake Flan-T5-Small Pruned 4-bit Encoder")\n    _stamp_metadata(dec_q, author, "OneTake Flan-T5-Small Pruned 4-bit Decoder")\n\n    enc_path = os.path.join(output_dir, "OneTakeParserEncoder.mlpackage")\n    dec_path = os.path.join(output_dir, "OneTakeParserDecoder.mlpackage")\n    for p in (enc_path, dec_path):\n        if os.path.exists(p):\n            shutil.rmtree(p)\n    enc_q.save(enc_path)\n    dec_q.save(dec_path)\n\n    artifacts = [enc_path, dec_path]\n\n    # Attempt Multi-Function Core ML merge into OneTakeParserANE.mlpackage to deduplicate shared.weight\n    ane_path = os.path.join(output_dir, "OneTakeParserANE.mlpackage")\n    if os.path.exists(ane_path):\n        shutil.rmtree(ane_path)\n    multi_fn_used = False\n    try:\n        desc = ct.utils.MultiFunctionDescriptor()\n        desc.add_function(enc_path, src_function_name="main", target_function_name="encode")\n        desc.add_function(dec_path, src_function_name="main", target_function_name="decode")\n        desc.default_function_name = "encode"\n        ct.utils.save_multifunction(desc, ane_path)\n        merged = ct.models.MLModel(ane_path, skip_model_load=True)\n        _stamp_metadata(merged, author, "OneTake Flan-T5-Small Multi-Function ANE Parser (4-bit)")\n        merged.save(ane_path)\n        shutil.rmtree(enc_path)\n        shutil.rmtree(dec_path)\n        artifacts = [ane_path]\n        multi_fn_used = True\n    except Exception as exc:\n        print(f"[OneTake] Multi-function merge fallback ({exc}); keeping Encoder + Decoder pair.")\n\n    total_bytes = sum(_dir_size_bytes(p) for p in artifacts)\n    if sp_vocab_path and os.path.exists(sp_vocab_path):\n        total_bytes += os.path.getsize(sp_vocab_path)\n\n    total_mb = total_bytes / (1024.0 * 1024.0)\n    print(f"[OneTake] Exported artifacts={artifacts}, multi_function={multi_fn_used}, total_size={total_mb:.2f} MB")\n    assert total_mb <= max_total_mb, f"Bundle size {total_mb:.2f} MB exceeded {max_total_mb} MB budget!"\n    return {\n        "artifacts": artifacts,\n        "multi_function": multi_fn_used,\n        "total_mb": round(total_mb, 2),\n    }\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--checkpoint-dir", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--sp-vocab", default=None)\n    parser.add_argument("--author", default="Porage-Lover <jitund05@gmail.com>")\n    parser.add_argument("--max-mb", type=float, default=35.0)\n    args = parser.parse_args()\n    export_coreml_models(\n        checkpoint_dir=args.checkpoint_dir,\n        output_dir=args.output_dir,\n        sp_vocab_path=args.sp_vocab,\n        author=args.author,\n        max_total_mb=args.max_mb,\n    )\n')
with open('/content/onetake_ml/test_dataset_generator.py', 'w', encoding='utf-8') as f:
    f.write('import unittest\nfrom dataset_generator import (\n    format_tagged_target,\n    generate_casting_corpus,\n    is_grounded_notice_value,\n)\n\n\nclass TestDatasetGenerator(unittest.TestCase):\n    def test_grounding_matches_swift_rule(self):\n        src = "MCCC - KOSHISHE\\nRole: Male Lead\\nEmail: abhinav@mcc.in\\n"\n        self.assertTrue(is_grounded_notice_value("MCCC - KOSHISHE", src))\n        self.assertTrue(is_grounded_notice_value("abhinav@mcc.in", src))\n        self.assertTrue(is_grounded_notice_value("Male Lead", src))\n        self.assertFalse(is_grounded_notice_value("fabricated@other.com", src))\n        self.assertFalse(is_grounded_notice_value("NONE", src))\n\n    def test_tagged_target_formats_none_cleanly(self):\n        out = format_tagged_target(\n            project="Photon",\n            role=None,\n            desc="Fresh male talent 22-25",\n            deadline=None,\n            email="cast@drama.com",\n        )\n        self.assertEqual(\n            out,\n            "PROJECT: Photon\\nROLE: NONE\\nDESC: Fresh male talent 22-25\\nDEADLINE: NONE\\nEMAIL: cast@drama.com",\n        )\n\n    def test_corpus_covers_traps_and_passes_grounding(self):\n        splits = generate_casting_corpus(num_train=240, num_val=60, num_test=60, seed=42)\n        self.assertEqual(len(splits["train"]), 240)\n        self.assertEqual(len(splits["val"]), 60)\n        self.assertEqual(len(splits["test"]), 60)\n\n        categories = {row["category"] for row in splits["train"]}\n        expected_categories = {\n            "standard_labeled",\n            "dual_email_trap",\n            "dual_date_trap",\n            "missing_fields",\n            "whatsapp_prose",\n            "trailing_sides",\n            "noisy_ocr",\n        }\n        self.assertTrue(expected_categories.issubset(categories))\n\n        for split_name, rows in splits.items():\n            for idx, row in enumerate(rows):\n                lines = row["target"].splitlines()\n                self.assertEqual(len(lines), 5, f"{split_name}[{idx}] bad target: {row[\'target\']}")\n                prefixes = [line.split(":", 1)[0] for line in lines]\n                self.assertEqual(prefixes, ["PROJECT", "ROLE", "DESC", "DEADLINE", "EMAIL"])\n                for line in lines:\n                    key, val = line.split(":", 1)\n                    val = val.strip()\n                    if val != "NONE":\n                        self.assertTrue(\n                            is_grounded_notice_value(val, row["input"]),\n                            f"Ungrounded {key}={val!r} in {row[\'category\']}:\\n{row[\'input\']}",\n                        )\n\n\nif __name__ == "__main__":\n    unittest.main()\n')
with open('/content/onetake_ml/test_vocab_surgery.py', 'w', encoding='utf-8') as f:
    f.write('import unittest\nfrom vocab_surgery import (\n    build_pruned_piece_table,\n    unigram_decode_reference,\n    unigram_encode_reference,\n)\n\n\nclass TestVocabSurgery(unittest.TestCase):\n    def test_pruning_preserves_specials_and_all_256_byte_tokens(self):\n        mock_pieces = [("<pad>", 0.0), ("</s>", 0.0), ("<unk>", 0.0)]\n        mock_pieces += [(f"<0x{b:02X}>", -20.0) for b in range(256)]\n        mock_pieces += [(f"\\u2581word{i}", -float((i % 10) + 1)) for i in range(500)]\n        mock_pieces += [(f"\\u2581unused_{i}", -8.0) for i in range(100)]\n        corpus = ["PROJECT: word1 word2", "ROLE: word3 \\u2013 Mumbai \\u20b95000"]\n\n        res = build_pruned_piece_table(mock_pieces, corpus, target_vocab_size=400)\n        vocab = res["vocab"]\n        self.assertEqual(len(vocab), 400)\n        self.assertEqual(vocab[0][0], "<pad>")\n        self.assertEqual(vocab[1][0], "</s>")\n        self.assertEqual(vocab[2][0], "<unk>")\n        pieces_set = {p for p, _ in vocab}\n        for b in range(256):\n            self.assertIn(f"<0x{b:02X}>", pieces_set)\n        self.assertIn("\\u2581word1", pieces_set)\n        self.assertIn("\\u2581word2", pieces_set)\n        self.assertEqual(len(res["old_to_new_ids"]), 400)\n\n    def test_reference_viterbi_round_trips_unicode_losslessly(self):\n        mock_pieces = [("<pad>", 0.0), ("</s>", 0.0), ("<unk>", 0.0)]\n        mock_pieces += [(f"<0x{b:02X}>", -20.0) for b in range(256)]\n        mock_pieces += [\n            ("\\u2581", -2.5),\n            ("\\u2581PROJECT", -1.5),\n            (":", -1.0),\n            ("\\u2581Male", -1.8),\n            ("\\u2581Lead", -1.9),\n        ]\n        res = build_pruned_piece_table(mock_pieces, ["PROJECT: Male Lead"], target_vocab_size=264)\n        sample = "PROJECT: Male Lead \\u2013 \\u20b95000 हिन्दी\\nabhinav@mcc.in"\n        expected = "PROJECT: Male Lead - Rs.5000 हिन्दी\\nabhinav@mcc.in"\n        ids = unigram_encode_reference(sample, res["vocab"])\n        self.assertEqual(ids[-1], 1)  # EOS token\n        decoded = unigram_decode_reference(ids, res["vocab"])\n        self.assertEqual(decoded, expected)\n\n    def test_flan_t5_non_byte_vocab_uses_extra_id_0_for_newlines(self):\n        mock_pieces = [("<pad>", 0.0), ("</s>", 0.0), ("<unk>", 0.0)]\n        mock_pieces += [(ch, -1.5) for ch in ["P", "R", "O", "J", "E", "C", "T", "L", "D", "A", "I", "M", "N", "2", "0", "5", ":", "-", ".", "@", "\\u2581"]]\n        mock_pieces += [\n            ("\\u2581PROJECT", -1.2),\n            ("\\u2581ROLE", -1.2),\n            ("\\u2581DESC", -1.2),\n            ("\\u2581DEADLINE", -1.2),\n            ("\\u2581EMAIL", -1.2),\n            ("\\u2581Male", -1.4),\n            ("\\u2581Lead", -1.4),\n            ("\\u258120", -1.4),\n            ("25", -1.4),\n            ("\\u2581NONE", -1.4),\n            ("<extra_id_0>", -10.0),\n            ("<extra_id_1>", -10.0),\n        ]\n        res = build_pruned_piece_table(mock_pieces, ["PROJECT: Male Lead"], target_vocab_size=35)\n        pieces_set = {p for p, _ in res["vocab"]}\n        self.assertIn("<extra_id_0>", pieces_set)\n        self.assertNotIn("<extra_id_1>", pieces_set)\n\n        tagged = "PROJECT: Male\\nROLE: Lead\\nDESC: 20\\u201325\\nDEADLINE: NONE\\nEMAIL: NONE"\n        ids = unigram_encode_reference(tagged, res["vocab"])\n        self.assertNotIn(2, ids)  # Zero <unk> collisions on newline or en-dash\n        decoded = unigram_decode_reference(ids, res["vocab"])\n        self.assertEqual(decoded, "PROJECT: Male\\nROLE: Lead\\nDESC: 20-25\\nDEADLINE: NONE\\nEMAIL: NONE")\n\n\nif __name__ == "__main__":\n    unittest.main()\n')
with open('/content/onetake_ml/test_export_wrappers.py', 'w', encoding='utf-8') as f:
    f.write('import os\nimport sys\nimport unittest\n\nsys.path.insert(0, os.path.dirname(__file__))\n\ntry:\n    import torch\n    from transformers import T5Config, T5ForConditionalGeneration\n    HAS_TORCH_HF = True\nexcept Exception:\n    HAS_TORCH_HF = False\n\n\n@unittest.skipUnless(HAS_TORCH_HF, "torch/transformers not available in sandbox python")\nclass TestExportWrappers(unittest.TestCase):\n    def test_encoder_and_decoder_wrappers_trace_cleanly(self):\n        from export_coreml import T5DecoderWrapper, T5EncoderWrapper\n\n        cfg = T5Config(\n            vocab_size=320,\n            d_model=64,\n            d_kv=16,\n            d_ff=128,\n            num_layers=2,\n            num_decoder_layers=2,\n            num_heads=4,\n            tie_word_embeddings=False,\n        )\n        model = T5ForConditionalGeneration(cfg).eval()\n        enc = T5EncoderWrapper(model).eval()\n        dec = T5DecoderWrapper(model).eval()\n\n        enc_ids = torch.ones((1, 128), dtype=torch.int32)\n        enc_mask = torch.ones((1, 128), dtype=torch.int32)\n        traced_enc = torch.jit.trace(enc, (enc_ids, enc_mask))\n        hidden = traced_enc(enc_ids, enc_mask)\n        self.assertEqual(tuple(hidden.shape), (1, 128, 64))\n\n        dec_ids = torch.zeros((1, 16), dtype=torch.int32)\n        traced_dec = torch.jit.trace(dec, (dec_ids, hidden, enc_mask))\n        logits = traced_dec(dec_ids, hidden, enc_mask)\n        self.assertEqual(tuple(logits.shape), (1, 16, 320))\n\n\nif __name__ == "__main__":\n    unittest.main()\n')


In [ ]:
!python3 -m unittest discover -s /content/onetake_ml -p 'test_*.py' -v
from train_and_qat import train_pipeline
metrics = train_pipeline(
    output_dir='/content/onetake_work',
    vocab_size=8192,
    num_train=18000,
    num_val=1000,
    epochs=5,
    qat_epochs=1,
    batch_size=32,
)
print('Training complete! Metrics:', metrics)


In [ ]:
from export_coreml import export_coreml_models
export_info = export_coreml_models(
    checkpoint_dir='/content/onetake_work/hf_checkpoint',
    output_dir='/content/onetake_bundle',
    sp_vocab_path='/content/onetake_work/sp_vocab.json',
    author='Porage-Lover <jitund05@gmail.com>',
    max_total_mb=35.0,
)
print('Core ML Export Result:', export_info)


In [ ]:
import shutil
shutil.copy('/content/onetake_work/sp_vocab.json', '/content/onetake_bundle/sp_vocab.json')
shutil.copy('/content/onetake_work/golden_parity_vectors.json', '/content/onetake_bundle/golden_parity_vectors.json')
shutil.copy('/content/onetake_work/eval_metrics.json', '/content/onetake_bundle/eval_metrics.json')
!du -sh /content/onetake_bundle/*
shutil.make_archive('/content/OneTakeParserCoreML_Bundle', 'zip', '/content/onetake_bundle')
try:
    from google.colab import files
    files.download('/content/OneTakeParserCoreML_Bundle.zip')
except ImportError:
    print('Zip ready at /content/OneTakeParserCoreML_Bundle.zip')
